# Analyze feature contributions in held-out test set -- compare to exploratory analyses in train set

- Salience
- LMG Feature Importance

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    get_test_df_path,
    get_perm_seed,
    col_config_sensitivity,
    export_data,
)
from src.sig import prep_train_test_model
from src.residual_func import get_saliency_map
from src.lmg_imp import get_confirm_lmg


import joblib
from functools import reduce
import pandas as pd

model_dir = BASE_PATH / "models"
df_dir = BASE_PATH / "data/test_data/cleaned"
res_export_path = BASE_PATH / "results/test"

## Salience

In [ ]:
for impact_type in ["self", "parent"]:
    model_params = BEST_PARAM_DICT["SQT"][impact_type]
    model_SQT = joblib.load(model_dir / f"SQT_{impact_type}.joblib")
    coef_df = model_SQT.get_coef_df()
    coef_df["stage_feat"] = coef_df["stage"] + "_" + coef_df["feature"]
    coef_df = coef_df.drop(["stage", "feature"], axis=1)
    coef_df = coef_df.set_index("stage_feat")
    coef_df = coef_df.rename(columns={"coef": f"coef_{impact_type}"})
    test_df_path = get_test_df_path(
        df_dir,
        llm_model=model_params["llm_model"],
        agg_type=model_params["agg_type"],
        desc_seg_type=model_params["desc_seg_type"],
        impact_type=impact_type,
        temp=model_params["temp"],
    )
    test_df = pd.read_parquet(test_df_path)
    model_params["impact_type"] = impact_type
    summary = get_saliency_map(
        config_dict=model_params,
        model=model_SQT,
        df=test_df,
        coef_df=coef_df,
    )
    display(summary)
    export_data(
        data_to_export=summary,
        export_path=res_export_path / f"tables/saliency/{impact_type}.tsv",
        remove_existing=True,
    )

## LMG

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = df_dir
model_dir = BASE_PATH / "models"
all_perm_dir = all_perm_dir = BASE_PATH / "results/perm_res/preds"
START_PERM = 0
END_PERM = 1000
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}

In [ ]:
res_dict = {}
for impact_type in ["parent", "self"]:
    if impact_type != "parent":
        continue
    print(f"{impact_type}...")
    for setup in ["SQT", "QT"]:
        print(f"\tSetup: {setup}")
        model_params = BEST_PARAM_DICT[setup][impact_type]
        X_train, y_train, X_test, y_test, model = prep_train_test_model(
            selected_params=model_params,
            impact_type=impact_type,
            train_data_dir=train_data_dir,
            test_data_dir=test_data_dir,
            shared_dict=SHARED_DICT,
        )
        model_config_dict = {
            **{
                k: v
                for k, v in model_params.items()
                if k not in ["outcome_sub_list", "llm_model", "temp"]
            },
            "quest_cols": col_config_sensitivity[f"{impact_type}_quest"],
            "impact_type": impact_type,
            **SHARED_DICT,
        }
        final_df = get_confirm_lmg(
            parent_perm_dir=all_perm_dir,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            model_config_dict=model_config_dict,
            start_perm=START_PERM,
            end_perm=END_PERM,
        )
        res_dict[impact_type] = final_df
        display(final_df)
        export_data(
            data_to_export=final_df,
            export_path=res_export_path / f"tables/LMG/{impact_type}_{setup}.tsv",
            remove_existing=True,
        )

## Aggregate

In [ ]:
# cv_lmg = pd.read_csv(BASE_PATH / "results/feat_cont/cv/mean_summary.tsv", sep="\t")
# cv_parent = cv_lmg.drop("avg_mean_self", axis=1)
# cv_self = cv_lmg.drop("avg_mean_parent", axis=1)

# self_lmg = pd.read_csv(res_export_path / f"tables/LMG/self.tsv", sep="\t")
# parent_lmg = pd.read_csv(res_export_path / f"tables/LMG/parent.tsv", sep="\t")

Get top 5 from each of parent QT/SQT train/test (4 total)

In [ ]:
# top_n = 20
# ascending = False
# top_self = set(
#     self_lmg.sort_values("avg_increase", ascending=ascending).iloc[:top_n].index
# )
# top_parent = set(
#     parent_lmg.sort_values("avg_increase", ascending=ascending).iloc[:top_n].index
# )


# top_cv_parent = set(
#     cv_parent.sort_values("avg_mean_parent", ascending=ascending).iloc[:top_n][
#         "outcome"
#     ]
# )
# top_cv_self = set(
#     cv_self.sort_values("avg_mean_self", ascending=ascending).iloc[:top_n]["outcome"]
# )
# top_cv_parent.intersection(top_parent)

In [ ]:
# test_lmg_combined = pd.merge(
#     self_lmg.set_index("outcome")["avg_increase"],
#     parent_lmg.set_index("outcome")["avg_increase"],
#     on="outcome",
#     suffixes=("_self_test", "_parent_test"),
# )

# cv_lmg_combined = pd.merge(
#     cv_self.set_index("outcome"), cv_parent.set_index("outcome"), on="outcome"
# ).rename(
#     columns={
#         "avg_mean_self": "avg_increase_self_dev",
#         "avg_mean_parent": "avg_increase_parent_dev",
#     }
# )
# lmg_all_combined = pd.merge(test_lmg_combined, cv_lmg_combined, on='outcome')
# # Relative absolute value
# lmg_all_combined_rasv = (lmg_all_combined / lmg_all_combined.abs().sum() * 100).round(1)
# lmg_all_combined_rasv

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(4, 16))
# sns.heatmap(
#     lmg_all_combined_rasv,
#     cmap="RdBu_r",
#     center=0,
#     annot=True,
#     fmt=".1f",
#     linewidths=0.3,
#     cbar_kws={"label": "% of column |total|"},
#     ax=ax,
# )
# ax.set_ylabel("")
# fig.tight_layout()